# 👩‍💻 **Prompt LLMs for Structured Output + Simulated Function Use**

**Time Estimate:** 45 minutes

## 📋 **Overview**

This activity focuses on enhancing your ability to craft effective prompts for generating structured outputs such as JSON/XML, and simulating logical processes with Large Language Models (LLMs). These skills are vital for developing applications that require precise data manipulation, common in fields such as software development and data analysis.

In this activity, you will create and refine prompts to guide LLMs for predictable outputs, and simulate function calls to ensure practical data logic using free, accessible models.

## 🎯 **Learning Outcomes**

By the end of this lab, you will be able to:

- Design prompts that yield structured JSON/XML outputs.
- Simulate logical operations using LLM-generated outputs.
- Refine prompt engineering skills to ensure accuracy and consistency.

## Task 1: Setup and Basic JSON Generation [15 minutes]

In [1]:
# imports
import torch
from transformers import pipeline
import json
import re

1. Setup a free local LLM and create helper functions for JSON parsing.
2. Test basic structured output generation.

In [5]:
# Task 1
# your code here...

# Setup free local LLM
model_name = "gpt2"
generator = pipeline('text-generation',
                    model=model_name,
                    max_length=300,
                    pad_token_id=50256)

def clean_and_parse_json(text):
    """Helper function to extract and clean JSON from model output"""
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return {"error": "Could not parse JSON", "raw": json_str}
    return {"error": "No JSON found", "raw": text}

print("✅ Model and helper functions ready")

# Test basic JSON generation
test_prompt = """Create a JSON object with name and age:
{
  "name": "Alice",
  "age": 25
}

Create similar JSON for Bob, age 30:
{"""

response = generator(test_prompt, max_length=50, temperature=0.2, do_sample=True)
generated = response[0]['generated_text'][len(test_prompt):]
result = clean_and_parse_json("{" + generated)
print("Test result:", result)

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


✅ Model and helper functions ready
Test result: {'error': 'No JSON found', 'raw': '{  "name": "Bob",\n\n '}


🔍 **Practice**

Test your setup with a simple prompt. Consider:

- How well the model generates structured output.
- The quality of JSON formatting from smaller models.

✅ **Success Checklist**

- Model loads without errors.
- JSON parsing helper function works correctly.
- Basic structured output generation tested.

💡 **Key Points**

- Smaller models require more explicit prompting for structured output.
- JSON parsing tools help handle imperfect model outputs.
- Clear examples in prompts improve consistency.

❗ **Common Mistakes to Avoid**

- Using high temperature for structured tasks (use 0.1-0.3 instead).
- Not providing exact format examples in prompts.
- Forgetting to validate generated JSON structure.

## Task 2: Travel Itinerary JSON Generation [15 minutes]
Create prompts that generate well-structured JSON for travel itineraries.
1. Design a prompt with exact JSON structure examples.
2. Generate travel itineraries for multiple destinations.
3. Validate the consistency of generated JSON.

In [6]:
# Task 2
# your code here...

def generate_travel_itinerary(destination, days=3):
    """Generate structured travel itinerary JSON"""

    prompt = f"""Create a travel itinerary in JSON format for {destination}. Follow this exact structure:

{{
  "destination": "{destination}",
  "duration_days": {days},
  "departure_date": "2024-06-15",
  "activities": [
    "Visit landmarks",
    "Try local cuisine"
  ],
  "accommodations": "Hotel",
  "budget_estimate": "$500-1000"
}}

Generate similar JSON for {destination}:
{{"""

    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)

        generated_text = response[0]['generated_text']
        generated_part = generated_text[len(prompt):]
        json_content = "{" + generated_part

        return clean_and_parse_json(json_content)

    except Exception as e:
        return {"error": str(e), "destination": destination}

# Test with multiple destinations
destinations = ["Paris", "Tokyo", "New York"]

for dest in destinations:
    print(f"\n=== {dest.upper()} ITINERARY ===")
    itinerary = generate_travel_itinerary(dest)
    print(json.dumps(itinerary, indent=2))


=== PARIS ITINERARY ===
{
  "error": "No JSON found",
  "raw": "{\n\n\"destination\": \"Paris\",\n\n\"duration_days\": 3,\n\n"
}

=== TOKYO ITINERARY ===
{
  "error": "No JSON found",
  "raw": "{\n\n\"destination\": \"Tokyo\",\n\n\"duration_days\": 3,"
}

=== NEW YORK ITINERARY ===
{
  "error": "No JSON found",
  "raw": "{\n\n\"destination\": \"New York\",\n\n\"duration_days\": 3,\n"
}


🔍 **Practice**

Test different destinations and analyze the quality of generated content.

✅ **Success Checklist**

- JSON structure matches expected format consistently.
- All required fields are present in generated outputs.
- Content is relevant and realistic for each destination.

💡 **Key Points**

- Providing exact JSON structure examples improves output consistency.
- Lower temperature settings help maintain structure.
- Multiple examples in prompts can improve format adherence.

❗ **Common Mistakes to Avoid**

- Vague structure specifications in prompts.
- Inconsistent field naming across examples.
- Not testing with diverse input variations.

## Task 3: Function Simulation - Order Calculations [15 minutes]
Simulate mathematical function logic using natural language prompts.
1. Create step-by-step calculation prompts.
2. Test order total calculations with tax and shipping.
3. Extract and verify numerical results from LLM outputs.

In [7]:
# Task 3
# your code here ...

def simulate_order_calculation(items_list):
    """Simulate order total calculation using LLM"""

    items_text = "\n".join([f"- {item['name']}: ${item['price']} x {item['quantity']}"
                           for item in items_list])

    prompt = f"""Calculate the total cost step by step:

Items:
{items_text}

Example:
- Laptop: $800 x 2 = $1600
- Mouse: $25 x 1 = $25
Subtotal: $1600 + $25 = $1625
Tax (8%): $1625 x 0.08 = $130
Total: $1625 + $130 = $1755

Calculate for items above:
Calculation:"""

    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 80,
                           temperature=0.1,
                           do_sample=True,
                           pad_token_id=50256)

        generated_text = response[0]['generated_text']
        calculation = generated_text[len(prompt):].strip()

        # Extract total from calculation
        total_match = re.search(r'Total:\s*\$?([0-9,]+\.?[0-9]*)', calculation, re.IGNORECASE)
        extracted_total = total_match.group(1) if total_match else "Not found"

        return {
            "calculation_steps": calculation,
            "extracted_total": extracted_total,
            "items": items_list
        }

    except Exception as e:
        return {"error": str(e), "items": items_list}

# Test order calculation
test_order = [
    {"name": "Coffee", "price": 5.99, "quantity": 3},
    {"name": "Sandwich", "price": 8.50, "quantity": 2}
]

print("=== ORDER CALCULATION SIMULATION ===")
result = simulate_order_calculation(test_order)

if "error" not in result:
    print("Items:")
    for item in result["items"]:
        print(f"  {item['name']}: ${item['price']} x {item['quantity']}")
    print(f"\nLLM Calculation:\n{result['calculation_steps']}")
    print(f"\nExtracted Total: ${result['extracted_total']}")

    # Verify with actual calculation
    actual_total = sum(item['price'] * item['quantity'] for item in test_order)
    tax = actual_total * 0.08
    final_total = actual_total + tax
    print(f"Actual Total (with 8% tax): ${final_total:.2f}")
else:
    print(f"Error: {result['error']}")

# Enhanced function simulation example
def simulate_route_decision(routes_data):
    """Simulate route optimization decision-making"""

    routes_text = "\n".join([
        f"Route {i+1}: {route['name']} - Distance: {route['distance']}km, "
        f"Traffic: {route['traffic']}, Time: {route['time']}min, Cost: ${route['cost']}"
        for i, route in enumerate(routes_data)
    ])

    prompt = f"""Choose the optimal route based on distance, traffic, time, and cost.

Available Routes:
{routes_text}

Analysis:
1. Distance comparison:
2. Traffic impact:
3. Time efficiency:
4. Cost consideration:
5. Overall recommendation:

Decision:"""

    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)

        generated_text = response[0]['generated_text']
        decision = generated_text[len(prompt):].strip()

        # Extract recommended route
        route_match = re.search(r'Route\s+(\d+)', decision, re.IGNORECASE)
        recommended_route = route_match.group(1) if route_match else "Not specified"

        return {
            "analysis": decision,
            "recommended_route": recommended_route,
            "routes": routes_data
        }

    except Exception as e:
        return {"error": str(e), "routes": routes_data}

# Test route optimization
sample_routes = [
    {"name": "Highway Route", "distance": 45, "traffic": "Light", "time": 35, "cost": 12.50},
    {"name": "City Route", "distance": 32, "traffic": "Heavy", "time": 55, "cost": 8.75}
]

print("\n=== ROUTE OPTIMIZATION SIMULATION ===")
route_result = simulate_route_decision(sample_routes)

if "error" not in route_result:
    print("Available Routes:")
    for i, route in enumerate(route_result["routes"], 1):
        print(f"  Route {i}: {route['name']} - {route['distance']}km, {route['time']}min, ${route['cost']}")

    print(f"\nLLM Analysis:\n{route_result['analysis'][:200]}...")
    print(f"\nRecommended Route: {route_result['recommended_route']}")
else:
    print(f"Error: {route_result['error']}")

=== ORDER CALCULATION SIMULATION ===
Items:
  Coffee: $5.99 x 3
  Sandwich: $8.5 x 2

LLM Calculation:
Items:

- Coffee: $5.99 x 3

- Sandwich: $

Extracted Total: $Not found
Actual Total (with 8% tax): $37.77

=== ROUTE OPTIMIZATION SIMULATION ===
Available Routes:
  Route 1: Highway Route - 45km, 35min, $12.5
  Route 2: City Route - 32km, 55min, $8.75

LLM Analysis:
1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1....

Recommended Route: Not specified


✅ **Success Checklist**

- Calculation steps are logically sequenced and clear.
- Mathematical operations are performed accurately.
- Final totals can be extracted and verified against actual calculations.

💡 **Key Points**

- Very low temperature (0.1) improves mathematical accuracy.
- Step-by-step examples help guide logical processes.
- Always verify LLM calculations with actual mathematical computation.

❗ **Common Mistakes to Avoid**

- Using high temperature for mathematical tasks.
- Not providing clear calculation examples.
- Trusting LLM math without verification.

🚀 **Next Steps**

In the next module, you will learn how to extend these structured output skills to more complex applications like API integration, database operations, and multi-step workflow automation. This builds on the concepts of prompt engineering and structured data generation to enhance your ability to create robust AI-powered applications.

## 💻 Exemplar Solution

<details>    
<summary><strong>Click HERE to see an exemplar solution</strong></summary>

### Task 1 Solution
    
```python
# Setup free local LLM
model_name = "gpt2"
generator = pipeline('text-generation',
                    model=model_name,
                    max_length=300,
                    pad_token_id=50256)

def clean_and_parse_json(text):
    """Helper function to extract and clean JSON from model output"""
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return {"error": "Could not parse JSON", "raw": json_str}
    return {"error": "No JSON found", "raw": text}

print("✅ Model and helper functions ready")

# Test basic JSON generation
test_prompt = """Create a JSON object with name and age:
{
  "name": "Alice",
  "age": 25
}

Create similar JSON for Bob, age 30:
{"""

response = generator(test_prompt, max_length=50, temperature=0.2, do_sample=True)
generated = response[0]['generated_text'][len(test_prompt):]
result = clean_and_parse_json("{" + generated)
print("Test result:", result)
```

### Task 2 Solution
    
```python
def generate_travel_itinerary(destination, days=3):
    """Generate structured travel itinerary JSON"""
    
    prompt = f"""Create a travel itinerary in JSON format for {destination}. Follow this exact structure:

{{
  "destination": "{destination}",
  "duration_days": {days},
  "departure_date": "2024-06-15",
  "activities": [
    "Visit landmarks",
    "Try local cuisine"
  ],
  "accommodations": "Hotel",
  "budget_estimate": "$500-1000"
}}

Generate similar JSON for {destination}:
{{"""
    
    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        generated_part = generated_text[len(prompt):]
        json_content = "{" + generated_part
        
        return clean_and_parse_json(json_content)
        
    except Exception as e:
        return {"error": str(e), "destination": destination}

# Test with multiple destinations
destinations = ["Paris", "Tokyo", "New York"]

for dest in destinations:
    print(f"\n=== {dest.upper()} ITINERARY ===")
    itinerary = generate_travel_itinerary(dest)
    print(json.dumps(itinerary, indent=2))
```

### Task 3 Solution

```python
def simulate_order_calculation(items_list):
    """Simulate order total calculation using LLM"""
    
    items_text = "\n".join([f"- {item['name']}: ${item['price']} x {item['quantity']}"
                           for item in items_list])
    
    prompt = f"""Calculate the total cost step by step:

Items:
{items_text}

Example:
- Laptop: $800 x 2 = $1600
- Mouse: $25 x 1 = $25
Subtotal: $1600 + $25 = $1625
Tax (8%): $1625 x 0.08 = $130
Total: $1625 + $130 = $1755

Calculate for items above:
Calculation:"""
    
    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 80,
                           temperature=0.1,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        calculation = generated_text[len(prompt):].strip()
        
        # Extract total from calculation
        total_match = re.search(r'Total:\s*\$?([0-9,]+\.?[0-9]*)', calculation, re.IGNORECASE)
        extracted_total = total_match.group(1) if total_match else "Not found"
        
        return {
            "calculation_steps": calculation,
            "extracted_total": extracted_total,
            "items": items_list
        }
        
    except Exception as e:
        return {"error": str(e), "items": items_list}

# Test order calculation
test_order = [
    {"name": "Coffee", "price": 5.99, "quantity": 3},
    {"name": "Sandwich", "price": 8.50, "quantity": 2}
]

print("=== ORDER CALCULATION SIMULATION ===")
result = simulate_order_calculation(test_order)

if "error" not in result:
    print("Items:")
    for item in result["items"]:
        print(f"  {item['name']}: ${item['price']} x {item['quantity']}")
    print(f"\nLLM Calculation:\n{result['calculation_steps']}")
    print(f"\nExtracted Total: ${result['extracted_total']}")
    
    # Verify with actual calculation
    actual_total = sum(item['price'] * item['quantity'] for item in test_order)
    tax = actual_total * 0.08
    final_total = actual_total + tax
    print(f"Actual Total (with 8% tax): ${final_total:.2f}")
else:
    print(f"Error: {result['error']}")

# Enhanced function simulation example
def simulate_route_decision(routes_data):
    """Simulate route optimization decision-making"""
    
    routes_text = "\n".join([
        f"Route {i+1}: {route['name']} - Distance: {route['distance']}km, "
        f"Traffic: {route['traffic']}, Time: {route['time']}min, Cost: ${route['cost']}"
        for i, route in enumerate(routes_data)
    ])
    
    prompt = f"""Choose the optimal route based on distance, traffic, time, and cost.

Available Routes:
{routes_text}

Analysis:
1. Distance comparison:
2. Traffic impact:
3. Time efficiency:
4. Cost consideration:
5. Overall recommendation:

Decision:"""
    
    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        decision = generated_text[len(prompt):].strip()
        
        # Extract recommended route
        route_match = re.search(r'Route\s+(\d+)', decision, re.IGNORECASE)
        recommended_route = route_match.group(1) if route_match else "Not specified"
        
        return {
            "analysis": decision,
            "recommended_route": recommended_route,
            "routes": routes_data
        }
        
    except Exception as e:
        return {"error": str(e), "routes": routes_data}

# Test route optimization
sample_routes = [
    {"name": "Highway Route", "distance": 45, "traffic": "Light", "time": 35, "cost": 12.50},
    {"name": "City Route", "distance": 32, "traffic": "Heavy", "time": 55, "cost": 8.75}
]

print("\n=== ROUTE OPTIMIZATION SIMULATION ===")
route_result = simulate_route_decision(sample_routes)

if "error" not in route_result:
    print("Available Routes:")
    for i, route in enumerate(route_result["routes"], 1):
        print(f"  Route {i}: {route['name']} - {route['distance']}km, {route['time']}min, ${route['cost']}")
    
    print(f"\nLLM Analysis:\n{route_result['analysis'][:200]}...")
    print(f"\nRecommended Route: {route_result['recommended_route']}")
else:
    print(f"Error: {route_result['error']}")
```
</details>